# Communication, Stakeholders, Ethics, Reproducibility and Taking Work to Production

Notebook for **Chapter 13** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch13.html

Tip: change a number, predict the output, then run the cell again.

## 13.1 Communicating Results: Answer First, Uncertainty Honest

In [ ]:
feats = ["rec", "logf", "pro", "ads"]
mu, sd = X.mean(0), X.std(0)
lr = LogisticRegression().fit((X - mu) / sd, y)
p = lr.predict_proba((X - mu) / sd)[:, 1]
i = int(np.argmin(np.abs(p - 0.31)))
print("cust", d.customer_id[i], "p", p[i].round(3), d.loc[i, feats].to_dict())
x = X[i].copy(); x[0] -= 10                       # she ordered 10 days more recently
print("rec - 10 days:", lr.predict_proba(((x - mu) / sd)[None, :])[0, 1].round(3))
x = X[i].copy(); x[1] = np.log1p(d.f[i] + 3)       # she has 3 more orders in history
print("+3 orders   :", lr.predict_proba(((x - mu) / sd)[None, :])[0, 1].round(3))
print("odds ratio per 10 days:", np.exp(lr.coef_[0][0] / sd[0] * 10).round(3))

## 13.2 Stakeholder Management and Working in Small Iterations

## 13.3 Ethics and Fairness in Practice

In [ ]:
d["group"] = np.where(d.city.isin(["Pune", "Chennai"]), "B", "A")
s = p >= 0.222
for gr in ["A", "B"]:
    m = (d.group == gr).values
    print(gr, "n", m.sum(), "selection rate", round(s[m].mean(), 3),
          "TPR", round(s[m & (y == 1)].mean(), 3), "FPR", round(s[m & (y == 0)].mean(), 3))
print("churn rate by group", d.groupby("group").churn.mean().round(3).to_dict())

A n 2052 selection rate 0.305 TPR 0.793 FPR 0.143
B n 510 selection rate 0.324 TPR 0.757 FPR 0.159
churn rate by group {'A': 0.249, 'B': 0.275}


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
gy = (d.group == "B").astype(int).values
pp = cross_val_predict(LogisticRegression(), (X - mu) / sd, gy, cv=5, method="predict_proba")[:, 1]
print("proxy AUC, model features -> group:", round(roc_auc_score(gy, pp), 3))
C = pd.get_dummies(d.city).values.astype(float)
pp2 = cross_val_predict(LogisticRegression(), np.c_[(X - mu) / sd, C], gy, cv=5, method="predict_proba")[:, 1]
print("proxy AUC, with city added:", round(roc_auc_score(gy, pp2), 3))

proxy AUC, model features -> group: 0.485
proxy AUC, with city added: 1.0


## 13.4 Reproducibility: One Command, Same Answer

In [ ]:
def run(seed):
    r = np.random.default_rng(seed)
    return r.normal(size=5).round(3).tolist()
print(run(1)); print(run(1)); print(run(2))

[0.346, 0.822, 0.33, -1.303, 0.905]
[0.346, 0.822, 0.33, -1.303, 0.905]
[0.189, -0.523, -0.413, -2.441, 1.8]


In [ ]:
import hashlib
h = hashlib.sha256(pd.util.hash_pandas_object(d[["customer_id", "churn"]], index=False).values.tobytes()).hexdigest()[:16]
print("data fingerprint", h)

data fingerprint caebcf2047b44370


## 13.5 Prototype to Production, Monitoring and Handover

In [ ]:
# app.py   run with:  streamlit run app.py        (standard usage, not run here)
import streamlit as st, joblib, numpy as np
model = joblib.load("models/churn_v1.joblib")
st.title("Churn risk")
rec = st.slider("Days since last order", 0, 90, 26)
orders = st.number_input("Past orders", 1, 200, 14)
pro = st.checkbox("Pro plan"); ads = st.checkbox("Acquired via ads")
p = model.predict_proba([[rec, np.log1p(orders), int(pro), int(ads)]])[0, 1]
st.metric("Churn probability", f"{p:.1%}")

In [ ]:
# api.py   run with:  uvicorn api:app             (standard usage, not run here)
from fastapi import FastAPI
from pydantic import BaseModel
import joblib, numpy as np
app = FastAPI(); model = joblib.load("models/churn_v1.joblib")
class Customer(BaseModel):
    rec: float; n_orders: int; pro: int; ads: int
@app.post("/score")
def score(c: Customer):
    return {"p_churn": float(model.predict_proba([[c.rec, np.log1p(c.n_orders), c.pro, c.ads]])[0, 1])}

In [ ]:
d["rec_bad"] = (END_DATE - d["last"]).dt.days             # serving bug: wrong reference date
Xs = d[feats].copy(); Xs["rec"] = d.rec_bad
ps = lr.predict_proba((Xs.values - mu) / sd)[:, 1]
print("mean days since last order:", d.rec.mean().round(1), "vs", d.rec_bad.mean().round(1))
print("mean risk:", p.mean().round(3), "vs", ps.mean().round(3))
print("share flagged:", (p >= .222).mean().round(3), "vs", (ps >= .222).mean().round(3))

mean days since last order: 19.4 vs 138.4
mean risk: 0.254 vs 0.998
share flagged: 0.309 vs 1.0


## Chapter recap

## Practice set

**1.** (Easy) Rewrite this as an answer-first sentence: "We analysed 12 months of data and built three models; XGBoost had AUC 0.91; the best threshold is 0.30." Add what is missing. **Answer:** Start with the decision and money: "Send the offer to customers with risk above 30%; we expect X rupees profit, assuming Y save rate." AUC moves to the appendix. What is missing is the money, the assumption and the risk.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch13.html